# Level 6 — Duplicate analysis

The biggest lesson here: a repeated customer/order is not the same as a duplicated transaction. I need to check the business key.

In [1]:
from pathlib import Path
import pandas as pd

# Works if I open Jupyter from the repo root or from notebooks/
data_path = Path("data/sales_raw.csv")
if not data_path.exists():
    data_path = Path("../data/sales_raw.csv")

df = pd.read_csv(data_path)
df.head(3)

,line_id,order_id,order_date,customer_id,city,category,product,quantity,unit_price_pkr,discount_pct,revenue_pkr,payment_method,order_status,channel,delivery_days
0,LN-00001,ORD-2025-25001,2025-05-13,C-1023,Lahore,Office,Whiteboard,4,6000,10%,21600.0,Card,Delivered,Store,10.0
1,LN-00002,ORD-2025-25001,2025-08-19,C-1134,Islamabad,Home,Wall Clock,5,3100,15%,13175.0,Cash,Delivered,Store,8.0
2,LN-00003,ORD-2025-25002,2025-09-10,C-1052,Rawalpindi,NaN,Desk Lamp,3,2592,0%,7776.0,Bank Transfer,Delivered,Online,6.0


### Exact duplicates

In [2]:
print("Exact duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("line_id").head(12)

Exact duplicate rows: 3


,line_id,order_id,order_date,customer_id,city,category,product,quantity,unit_price_pkr,discount_pct,revenue_pkr,payment_method,order_status,channel,delivery_days
26,LN-00027,ORD-2025-25014,16/02/2025,C-1056,KHI,Home,Water Bottle,2,950,0%,1900.0,Cash,Delivered,Store,5.0
560,LN-00027,ORD-2025-25014,16/02/2025,C-1056,KHI,Home,Water Bottle,2,950,0%,1900.0,Cash,Delivered,Store,5.0
112,LN-00113,ORD-2025-25057,2025-09-21,C-1098,Lahore,Office,Whiteboard,4,6000,0%,24000.0,Card,Delivered,Store,3.0
561,LN-00113,ORD-2025-25057,2025-09-21,C-1098,Lahore,Office,Whiteboard,4,6000,0%,24000.0,Card,Delivered,Store,3.0
301,LN-00302,ORD-2025-25151,2025-02-08,C-1102,Rawalpindi,Accessories,Charging Cable,4,950,0%,3800.0,Wallet,Delivered,Online,11.0
562,LN-00302,ORD-2025-25151,2025-02-08,C-1102,Rawalpindi,Accessories,Charging Cable,4,950,0%,3800.0,Wallet,Delivered,Online,11.0


### An order can have multiple line items

In [3]:
print("Repeated order_id rows:", df.duplicated("order_id").sum())
example_order = df["order_id"].value_counts().idxmax()
df.loc[df["order_id"] == example_order,
       ["order_id", "line_id", "product", "quantity", "revenue_pkr"]]

Repeated order_id rows: 285


,order_id,line_id,product,quantity,revenue_pkr
242,ORD-2025-25122,LN-00243,Whiteboard,1,5760.0
243,ORD-2025-25122,LN-00244,Power Bank,2,14784.0
564,ORD-2025-25122,LN-00244,Power Bank,2,14784.0


### Check the actual line-level key

In [4]:
key_dupes = df[df.duplicated(subset=["line_id"], keep=False)].sort_values("line_id")
print("Rows sharing a line_id:", len(key_dupes))
key_dupes[["line_id", "order_id", "product", "revenue_pkr", "order_status"]]

Rows sharing a line_id: 10


,line_id,order_id,product,revenue_pkr,order_status
26,LN-00027,ORD-2025-25014,Water Bottle,1900.0,Delivered
560,LN-00027,ORD-2025-25014,Water Bottle,1900.0,Delivered
88,LN-00089,ORD-2025-25045,USB-C Hub,8064.0,Delivered
563,LN-00089,ORD-2025-25045,USB-C Hub,8564.0,Delivered
112,LN-00113,ORD-2025-25057,Whiteboard,24000.0,Delivered
561,LN-00113,ORD-2025-25057,Whiteboard,24000.0,Delivered
243,LN-00244,ORD-2025-25122,Power Bank,14784.0,Delivered
564,LN-00244,ORD-2025-25122,Power Bank,14784.0,Pending
301,LN-00302,ORD-2025-25151,Charging Cable,3800.0,Delivered
562,LN-00302,ORD-2025-25151,Charging Cable,3800.0,Delivered


### Exact copies vs conflicting records
I can remove obvious full-row copies for this exercise. But a repeated `line_id` with different values needs investigation.

In [5]:
clean_exact = df.drop_duplicates().copy()
print("Before:", len(df), "After exact dedup:", len(clean_exact))

still_conflicting = clean_exact[clean_exact.duplicated("line_id", keep=False)]
print("Rows with duplicated line IDs after exact dedup:", len(still_conflicting))
still_conflicting[["line_id", "revenue_pkr", "order_status"]].sort_values("line_id")

Before: 565 After exact dedup: 562
Rows with duplicated line IDs after exact dedup: 4


,line_id,revenue_pkr,order_status
88,LN-00089,8064.0,Delivered
563,LN-00089,8564.0,Delivered
243,LN-00244,14784.0,Delivered
564,LN-00244,14784.0,Pending


### A quick quality check

In [6]:
checks = {
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "line_ids_with_conflicts": int(still_conflicting["line_id"].nunique()),
    "unique_customers": int(df["customer_id"].nunique()),
    "repeated_customer_rows_not_automatically_wrong": int(df.duplicated("customer_id").sum())
}
pd.Series(checks)

exact_duplicate_rows                                3
line_ids_with_conflicts                             2
unique_customers                                  172
repeated_customer_rows_not_automatically_wrong    392
dtype: int64

**My note:** I should never call `drop_duplicates(subset=["customer_id"])` on a sales line dataset. One customer can order many times. Conflicting business keys should be investigated before dropping records.